# Pipeline Hoàn chỉnh: Từ Dữ liệu đến Submission

Notebook này thực thi toàn bộ quy trình tích hợp:
- `PROFILE = 'baseline'`: Trích xuất 32 đặc trưng thống kê, huấn luyện Logistic Regression 3-fold CV, đánh giá OOF và xuất file submission nếu có test set.
- `PROFILE = 'full'`: Huấn luyện toàn bộ LR + 5 cấu hình CNN x 3 fold (`frozen`, `center60`, `native`, `resampled`, `b2`), tổng hợp bảng so sánh OOF, thực hiện Blending 50/50 B2/Native và xuất submission.
- Nếu thiếu dữ liệu train: Báo lỗi dừng sớm ngay từ đầu.
- Nếu thiếu tập test: Hoàn thành đánh giá development và ghi nhận trạng thái `no submission produced`.

In [ ]:
from pathlib import Path
import os
import sys

def find_task_root():
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, cwd / 'KeMaoDanh', cwd.parent / 'KeMaoDanh']:
        if (cand / 'src/kmd').is_dir() and (cand / 'configs').is_dir():
            return cand.resolve()
    raise FileNotFoundError("Mở notebook từ repo root, KeMaoDanh hoặc KeMaoDanh/notebooks.")

TASK_ROOT = find_task_root()
if str(TASK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(TASK_ROOT / 'src'))

import torch
import numpy as np
import pandas as pd

from kmd.core import PACKAGE, read_csv, read_json, write_json, metric, export_submission
from kmd.pipeline import (
    prepare_development, start_session, fit_lr_cv, train_cnn_cv,
    compare_oof, blend, load_pairs, check_test_separation, infer_cnn, infer_lr, aligned_predictions
)

# === CẤU HÌNH THỰC THI ===
PROFILE = 'baseline'  # 'baseline' hoặc 'full'
RUN_ID = 'end_to_end_session'

DATA_ROOT = os.environ.get('DATA_ROOT') or str(TASK_ROOT / 'data/train')
TEST_ROOT = os.environ.get('TEST_ROOT') or str(TASK_ROOT / 'data/test')

print(f"PROFILE: {PROFILE} | RUN_ID: {RUN_ID}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"TEST_ROOT: {TEST_ROOT}")
print(f"CUDA: {torch.cuda.is_available()}")

if PROFILE == 'full' and not torch.cuda.is_available():
    raise RuntimeError("PROFILE='full' bắt buộc phải có GPU CUDA. Hãy chuyển sang PROFILE='baseline' để chạy trên CPU.")
if PROFILE not in {'baseline', 'full'}:
    raise ValueError("PROFILE chỉ nhận 'baseline' hoặc 'full'.")

## 1. Nạp dữ liệu 800 Development và Khởi tạo Session

In [ ]:
train_path = Path(DATA_ROOT).expanduser().resolve()
if not (train_path / 'pairs.csv').is_file():
    raise FileNotFoundError(f"Không tìm thấy {train_path / 'pairs.csv'}. Vui lòng kiểm tra lại DATA_ROOT.")

dev_frame = prepare_development(train_path)
session_dir = start_session(dev_frame, train_path, run_id=RUN_ID)
print(f"Khởi tạo session tại: {session_dir.relative_to(PACKAGE)}")

test_path = Path(TEST_ROOT).expanduser().resolve()
has_test = (test_path / 'pairs.csv').is_file()
test_manifest = None
if os.environ.get('TEST_ROOT') and not has_test:
    raise FileNotFoundError(f'TEST_ROOT đã đặt nhưng thiếu pairs.csv: {test_path}')

if has_test:
    test_manifest = load_pairs(test_path, labeled=False)
    check_test_separation(test_manifest, test_path, session_dir)
    print(f"Nạp {len(test_manifest)} cặp ảnh test thành công.")
else:
    print("Không tìm thấy bộ test. Pipeline sẽ chỉ huấn luyện và đánh giá trên tập Development.")

## 2. Huấn luyện 3-Fold Logistic Regression

In [ ]:
print("Huấn luyện 3-Fold Logistic Regression (32 features)...")
lr_oof = fit_lr_cv(dev_frame, train_path, session_dir)
score_lr = metric(dev_frame.fake_position, lr_oof.p)

print(f"Kết quả LR 3-Fold OOF: Macro-F1 = {score_lr['macro_f1']:.4f}, Accuracy = {score_lr['accuracy']:.4f}, Số lỗi = {score_lr['errors']}/800")

## 3. Huấn luyện 5 Cấu hình CNN x 3 Folds (Khi `PROFILE == 'full'`)

In [ ]:
models_to_eval = ['lr']

if PROFILE == 'full':
    cnn_names = ['frozen', 'center60', 'native', 'resampled', 'b2']
    for name in cnn_names:
        print(f"Huấn luyện 3-Fold CNN: {name}...")
        train_cnn_cv(name, dev_frame, train_path, session_dir)
    models_to_eval.extend(cnn_names)

comp_df, oof_dict = compare_oof(session_dir, models_to_eval)
if PROFILE == 'full':
    blend_oof = blend(oof_dict['b2'], oof_dict['native'], dev_frame)
    blend_oof.to_csv(session_dir / 'blend_oof.csv', index=False)
    oof_dict['blend'] = blend_oof
    comp_df = pd.concat([comp_df, pd.DataFrame([{'model': 'blend', **metric(dev_frame.fake_position, blend_oof.p)}])], ignore_index=True)
comp_df.to_csv(session_dir / 'comparison.csv', index=False)
print("\n=== BẢNG TỔNG HỢP SO SÁNH OUT-OF-FOLD (800 MẪU) ===")
print(comp_df.to_string(index=False))

## 4. Kết hợp Blending và Xuất file Submission

In [ ]:
if has_test:
    out_dir = PACKAGE / 'outputs' / session_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)
    
    if PROFILE == 'full':
        print("Inference test bằng mô hình Blend 50% B2 + 50% Native...")
        p_b2 = infer_cnn('b2', test_manifest, test_path, session_dir)
        p_nat = infer_cnn('native', test_manifest, test_path, session_dir)
        final_pred = blend(p_b2, p_nat, test_manifest)
    else:
        print("Inference test bằng mô hình 3-Fold Logistic Regression...")
        final_pred = infer_lr(test_manifest, test_path, session_dir)
        
    final_pred = aligned_predictions(final_pred, test_manifest)
    final_pred.to_csv(out_dir / 'test_probabilities.csv', index=False)
    sub_df = export_submission(final_pred, out_dir / 'submission.csv')
    print(f"Đã tạo file submission tại: {out_dir / 'submission.csv'}")
    print(sub_df.head())
else:
    print("Notice: Missing or unprovided test-root. Development training/evaluation completed successfully.")
    print("Status: no submission produced (test root not provided).")